# 04. Advanced Parsing with Knowledge

**Purpose:** Capstone challenge (Task 4: Regional Complexity). Build a two-stage pipeline that detects the country and retrieves country-specific rules from a knowledge base.

**Key Takeaway:** Some problems require smarter system design (RAG-style retrieval + structured prompting), not just bigger models.


## Step 1: Install Dependencies

In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git
%pip install -q -r esmt-workshop/requirements.txt


## Step 2: Load Shared Utilities

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/esmt-workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses, process_single_address
print('PROJECT_ROOT =', PROJECT_ROOT)


## Step 3: Student Identity and Generation Controls

In [ ]:
from google.colab import auth

auth.authenticate_user()

gcloud_email = !gcloud config get-value account
print(f"Authenticated as: {gcloud_email[0] if gcloud_email else 'Unknown'}")
os.environ['WORKSHOP_EMAIL'] = gcloud_email[0] if gcloud_email else os.environ.get('WORKSHOP_EMAIL','')

# Students only provide email; proxy endpoint details are managed by organizers.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', 'student@example.com')
print("STUDENT_EMAIL =", STUDENT_EMAIL)


In [ ]:
# Notebook metadata used in experiment history logs.
NOTEBOOK_NAME = '04_advanced_parsing_with_knowledge'
RUN_NOTES = ''

# Stage must be one of: baseline, prompt_tuned, advanced, two_stage
STAGE_NAME = 'two_stage'

# Model selection (organizers may override via env vars).
MODEL_NAME = os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro')
COUNTRY_MODEL = os.getenv('WORKSHOP_COUNTRY_MODEL', '')  # optional

# Generation controls (keep defaults consistent with other notebooks).
TEMPERATURE = 0.0
TOP_P = 0.95
TOP_K = 40
MAX_TOKENS = None
MAX_WORKERS = 8

# Guardrails are optional for this capstone.
USE_GUARDRAILS = False

# Knowledge base CSV (country-specific address rules)
KB_CSV_PATH = str(PROJECT_ROOT / 'data/input/address_formats.csv')

# Offline-safe switch (set False only if gateway access is working).
MOCK_MODE = True

print("STAGE_NAME =", STAGE_NAME)
print("MODEL_NAME =", MODEL_NAME)
print("COUNTRY_MODEL =", COUNTRY_MODEL or "(none)")
print("MOCK_MODE =", MOCK_MODE)


## Step 4: Edit Prompt Directly in Notebook

In [ ]:
# Set to False to use built-in stage prompt logic.
USE_CUSTOM_PROMPT = True
PROMPT_LABEL = 'student_prompt'

PROMPT_TEMPLATE = '''You are an AML address parser.

You MUST return strict JSON only using this schema:
{schema}

You will be given an unstructured address. Your job is to extract:
- Town Name (city/locality only)
- Postal Code (postal token(s) only)
- Remaining Address (everything else)
- Country Code (2 characters) = ISO alpha-2 uppercase

Input address:
{address}

Rules:
- Do not invent values.
- Use empty strings when uncertain.
'''

custom_prompt = PROMPT_TEMPLATE if USE_CUSTOM_PROMPT else None


## Step 5: Run Two-Stage Pipeline on Unlabeled Test Set

In [ ]:
import time

# Unlabeled test set used for the capstone submission.
test_unlabeled = pd.read_csv(PROJECT_ROOT / 'data/workshop/test_unlabeled.csv', dtype=str).fillna('')
input_df = test_unlabeled

# Execute configured stage and measure runtime.
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    input_df,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=custom_prompt,
    kb_csv_path=KB_CSV_PATH,
    max_workers=MAX_WORKERS,
    mock_mode=MOCK_MODE,
)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))
pred_df.head(5)


## Step 6: Optional Local Validation

In [ ]:
# Optional local validation (if labels are available).
test_labeled_path = PROJECT_ROOT / 'data/workshop/test_labeled.csv'
if test_labeled_path.exists():
    test_labeled = pd.read_csv(test_labeled_path, dtype=str).fillna('')
    report = evaluate_predictions(pred_df, test_labeled, email=STUDENT_EMAIL)
    print(report['summary'])
    display(report['field_metrics'])
else:
    report = {'summary': {}}
    print('test_labeled.csv not found. Skipping evaluation.')


## Step 7: Save Artifacts and Log the Run

In [ ]:
out_dir = PROJECT_ROOT / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

pred_path = out_dir / '04_advanced_parsing_with_knowledge_predictions.csv'
pred_df.to_csv(pred_path, index=False)

# Submission file for manual upload (same data as predictions; name makes intent clear).
submission_path = out_dir / '04_final_submission.csv'
pred_df.to_csv(submission_path, index=False)

report_dir = out_dir / 'report_04_advanced_parsing_with_knowledge'
if report.get('summary'):
    save_evaluation_report(report, report_dir)

run_record = log_experiment_run(
    output_root=PROJECT_ROOT / 'outputs',
    notebook_name=NOTEBOOK_NAME,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    max_workers=MAX_WORKERS,
    use_guardrails=USE_GUARDRAILS,
    kb_csv_path=KB_CSV_PATH,
    prompt_template=custom_prompt,
    prompt_label=PROMPT_LABEL if USE_CUSTOM_PROMPT else 'stage_default_prompt',
    runtime_sec=runtime_sec,
    metrics_summary=report.get('summary', {}),
    notes=RUN_NOTES,
    predictions_path=pred_path,
    report_dir=report_dir if report.get('summary') else '',
)

print('Saved predictions:', pred_path)
print('Saved submission:', submission_path)
if report.get('summary'):
    print('Saved report:', report_dir)
print('Logged run_id:', run_record['run_id'])


## Step 8: Prompt/Parameter History Summary

In [ ]:
history_df = load_experiment_history(output_root=PROJECT_ROOT / 'outputs', notebook_name=NOTEBOOK_NAME)
display(history_df.tail(20))


## Assignment

1. Run the two-stage pipeline on the unlabeled test set (`test_unlabeled.csv`).
2. Inspect a few outputs and verify the schema looks correct.
3. If you have access to the labeled test set locally, review the evaluation report.
4. Submit `outputs/04_final_submission.csv` to the leaderboard (manual upload).

**Discussion prompts**
- Where did the two-stage approach help compared to a single prompt?
- What failure modes remain (e.g., ambiguous countries, incomplete addresses)?
- What would you add to the knowledge base to improve performance?